In [9]:
# Import required packages
using DifferentialEquations     # Main ODE solver framework
using LinearAlgebra             # For matrix operations (mul!)
using SparseArrays              # Sparse matrix handling
using KLU                       # Sparse LU factorization
using SparseDiffTools           # For matrix_colors 
using PreallocationTools        # For DiffCache
using DiffEqCallbacks           # For PositiveDomain callback
using SciMLSensitivity, Enzyme  # For automatic differentation
using ADTypes                   # For automatic differentation
using Plots                     # For plotting results
using YAML                      # For parsing YAML files

# Ensure all packages are installed. If not, instruct the user to install them.
# Example: Pkg.add("DifferentialEquations"), etc.

include("Chemistry.jl") # Chemistry.jl is the actual program

energy_accounting (generic function with 1 method)

In [10]:
# Set up the ODE problem
const FILENAME = "LuMechanism.yaml"

# Load the mechanism file
mechanism_data = load_mechanism_data(FILENAME)

# Process the chemical data
species_list, reaction_list, species_index_map = process_chemical_data(mechanism_data)

# Number of species and reactions
const n_species = length(species_list)
const n_vars = n_species + 1
const n_reactions = length(reaction_list)

# Build the stoichiometric matrix and kinetics list
S = sparse(build_stoichiometric_matrix(reaction_list, n_species))
kinetics_list = build_kinetics_list(reaction_list)

SplitKinetics(ElementaryKinetics[ElementaryKinetics(1.2e17, -1.0, 0.0, true, true, [(15, 1.75), (6, 15.4), (16, 3.6), (26, 3.0), (14, 2.0), (1, 2.4)], [3], [2.0], [4], [1.0]), ElementaryKinetics(5.0e17, -1.0, 0.0, true, true, [(15, 1.5), (6, 6.0), (16, 2.0), (26, 3.0), (14, 2.0), (1, 2.0)], [3, 2], [1.0, 1.0], [5], [1.0]), ElementaryKinetics(38700.0, 2.7, 6260.0, false, true, Tuple{Int64, Float64}[], [3, 1], [1.0, 1.0], [2, 5], [1.0, 1.0]), ElementaryKinetics(2.0e13, 0.0, 0.0, false, true, Tuple{Int64, Float64}[], [3, 7], [1.0, 1.0], [5, 4], [1.0, 1.0]), ElementaryKinetics(9.63e6, 2.0, 4000.0, false, true, Tuple{Int64, Float64}[], [3, 8], [1.0, 1.0], [5, 7], [1.0, 1.0]), ElementaryKinetics(5.7e13, 0.0, 0.0, false, true, Tuple{Int64, Float64}[], [3, 10], [1.0, 1.0], [2, 15], [1.0, 1.0]), ElementaryKinetics(8.0e13, 0.0, 0.0, false, true, Tuple{Int64, Float64}[], [3, 11], [1.0, 1.0], [2, 17], [1.0, 1.0]), ElementaryKinetics(1.5e13, 0.0, 0.0, false, true, Tuple{Int64, Float64}[], [3, 12], 

In [11]:
config = ChemistryConfig(
        temperature = 900.0, # Initial temperature in K
        pressure = 4e5,      # Initial pressure in Pa
        fuel_mixture = Dict("CH4" => 1/10.52), air_percentage = 9.52/10.52
)

X0 = initialize_concentrations(config, species_list, species_index_map)

31-element Vector{Float64}:
 900.0
   0.0
   0.0
   0.0
  10.221925140702425
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   ⋮
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
  38.106025144973174

In [12]:
function dT(X::Vector{Float64}, r::Vector{Float64}, S::SparseMatrixCSC{Float64, Int64}, species_list::Vector{Species})
    n_species = length(species_list)
    n_reactions = length(r)

    u0_vec = zeros(n_reactions)      # Enthalpy change per reaction (J/mol)
    cv_vec = zeros(n_species)        # Heat capacity per species (J/(mol·K))

    T = X[1]                         # Temperature in K
    concentrations = X[2:end]        # Species concentrations in mol/m³

    # Pre-compute species enthalpies and heat capacities
    u_species = zeros(n_species)     # Enthalpy for each species (J/mol)
    for (species_index, species) in enumerate(species_list)
        cv_vec[species_index] = species_cp(T, species.thermo) - R_joule
        u_species[species_index] = h0(T, species.thermo) - R_joule * T
    end

    # Compute h0_vec for reactions
    for reaction_index in 1:n_reactions
        # Sum over species: stoichiometric coefficient * species enthalpy
        for species_index in 1:n_species
            stoich_coeff = S[species_index, reaction_index]
            if stoich_coeff != 0.0
                u0_vec[reaction_index] += stoich_coeff * u_species[species_index]
            end
        end
    end

    # Compute the total heat change rate (J/(m³·s))
    Q = sum(r .* u0_vec)

    # Compute the total heat capacity of the mixture (J/(m³·K))
    c_v = sum(concentrations .* cv_vec)

    # Compute temperature rate of change (K/s)
    dT_dt = -Q / c_v

    return dT_dt
end
    
# Define the ODE function
function reaction_ode!(dX, X, p, t)
    # Unpack parameters
    S, kinetics_tuples, species_list, reaction_list, species_index_map = p

    X = max.(X,1e-50)

    r = zeros(length(reaction_list))

    # Compute reaction rates
    compute_reaction_rates!(r, X, kinetics_tuples, species_list)

    # Compute temperature rate of change
    dT_dt = dT(X, r, S, species_list)

    # Compute species concentration rate of change
    dC_dt = S * r  # dC/dt = S * r

    # Populate the derivative vector
    dX[1] = dT_dt       # Temperature derivative
    @views dX[2:end] .= dC_dt  # Species concentration derivatives
end

reaction_ode! (generic function with 1 method)

In [13]:
# Define Jacobian sparsity structure
function build_local_jacobian_sparsity(S)
    stoich_mat = S
    J = spzeros(Bool, n_vars, n_vars)
    for r in 1:size(stoich_mat, 2)
        participants = findall(!iszero, stoich_mat[:, r])
        for i in participants, j in participants
            J[i+1, j+1] = true
        end
    end
    for spec in 1:n_species
        J[spec+1, 1] = true
        J[1, spec+1] = true
    end
    J[1, 1] = true
    return J
end

build_local_jacobian_sparsity (generic function with 1 method)

In [14]:
# Define time span and solver settings
tspan = (0.0, 1)
abstol, reltol = 1e-12, 1e-9

# Set up the ODE problem
params = S, kinetics_list, species_list, reaction_list, species_index_map

J = build_local_jacobian_sparsity(S)
colorvec = matrix_colors(J)

ode_func = ODEFunction(reaction_ode!, sparsity=J, colorvec=colorvec)

problem = ODEProblem(ode_func, X0, tspan, params)

algo = Rodas4P(
    linsolve = KLUFactorization(),
    autodiff = AutoEnzyme(; function_annotation=Enzyme.Duplicated),
    standardtag = false,
    concrete_jac = true
) 

# Solve the ODE problem
@time sol = solve(problem, # algo not used <==========
            verbose=false,
            abstol=abstol,
            reltol=reltol)

2403.043045 seconds (1.35 G allocations: 148.505 GiB, 5.04% gc time, 1.12% compilation time)


retcode: MaxIters
Interpolation: 3rd order Hermite
t: 431085-element Vector{Float64}:
 0.0
 8.009363655882122e-11
 1.2587134264758904e-10
 5.836484035352671e-10
 5.161419012412048e-9
 2.1211169679914205e-8
 1.3627312526130778e-7
 2.965209303803987e-7
 3.154230097790564e-7
 3.292873934860918e-7
 3.7753440571668893e-7
 4.111033560552537e-7
 4.760454951684387e-7
 ⋮
 0.0019229314487230504
 0.0019229318353486207
 0.0019229357016043243
 0.0019229743641613602
 0.0019229752791771462
 0.001922977109208718
 0.0019229778412213467
 0.0019229781340263982
 0.0019229783526283901
 0.0019229783942848314
 0.0019229784083920733
 0.0019229784188093384
u: 431085-element Vector{Vector{Float64}}:
 [900.0, 0.0, 0.0, 0.0, 10.221925140702425, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 38.106025144973174]
 [899.9999999998911, 6.33694777875751e-19, 1.597132913626559e-12, 2.7158036523851384e-19, 10.221925140702417, 2.7100745731786267e-19, 3.831428590967472e-23, 7.074158439422103e-15, 

In [15]:
# Extract the solution arrays
t = sol.t
T = sol[1, :]                  # Temperature over time
concentrations = sol[2:end, :]'  # Species concentrations over time

idx = [species_index_map["CH4"],species_index_map["O2"],species_index_map["H2O"],species_index_map["CO2"],species_index_map["CO"],species_index_map["NO"],species_index_map["O"],species_index_map["OH"],species_index_map["H"],species_index_map["H2"]]

# Plot results
plot1 = plot(t, T,
         xlabel = "t (s)",
         ylabel = "T (K)",
         legend = false)

plot2 = plot(t, concentrations[:,idx], label=[species_list[i].name for i in idx'],
         xlabel = "t (s)",
         ylabel = "c (mol/m³)")

plot!(plot2, t, concentrations[:,setdiff(setdiff(1:n_species, idx), [species_index_map["N2"]])], label=false,
         xlabel = "t (s)",
         ylabel = "c (mol/m³)")

plot(plot1, plot2, layout = (2,1))
savefig("SingleNodeCH4")

LoadError: KeyError: key "NO" not found

In [16]:
using LinearAlgebra: norm

function compute_relaxation_time(sol; tol=1e-6)
    """
    Compute relaxation time τ for a solution `sol` of an ODEProblem.
    
    τ is defined as the time when ‖u(t) - u_steady‖ / ‖u0 - u_steady‖ ≈ 1/e.
    
    Args:
        sol: Solution object from DifferentialEquations.jl
        tol: Tolerance for checking convergence (default: 1e-6)
    
    Returns:
        τ: Relaxation time (first time when decay reaches 1/e)
        If no such time is found, returns `nothing`.
    """
    u0 = sol.prob.u0          # Initial (perturbed) state
    u_steady = sol[end]       # Equilibrium state (final value)
    Δ0 = norm(u0 - u_steady)  # Initial deviation magnitude
    
    # Handle cases where Δ0 ≈ 0 (no perturbation)
    if Δ0 < tol
        @warn "Initial state is already at equilibrium (‖Δu‖ = $Δ0). τ is undefined."
        return nothing
    end
    
    # Target deviation: Δ0 / e
    target_deviation = Δ0 / MathConstants.e
    
    # Find the first time when ‖u(t) - u_steady‖ ≤ target_deviation
    τ = nothing
    for (i, t) in enumerate(sol.t)
        Δu = norm(sol.u[i] - u_steady)
        if Δu ≤ target_deviation + tol  # Allow numerical tolerance
            τ = t
            break
        end
    end
    
    if τ === nothing
        @warn "No relaxation time found within solution timeframe. Increase `tspan`."
    end
    
    return τ
end

compute_relaxation_time(sol)

0.0008566515534273597